In [49]:
#import packages and fech api keys from env file
import os
import sys
sys.path.append("..")
from util import check_api_key
from dotenv import load_dotenv
from openai import OpenAI
import ollama
import anthropic
import gradio as gr 
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display, update_display
import json

openai_api_key = check_api_key("openai")
anthropic_api_key = check_api_key("anthropic")



Openai API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-a


In [50]:
#connect to openai and anthropic
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)


In [51]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [52]:
system_message = "You are a helpful assistant who responds in simple and easy to understand sentences. " \
"You respond in markdown without code blocks."

def stream_gpt(prompt):
    messages = [
        {"role" : "system", "content": brochure_system_prompt},
        {"role" : "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages,
        stream = True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=display_handle.display_id)
        #yield result


def stream_claude(prompt):
    messages = [
        {"role" : "system", "content" : brochure_system_prompt},
        {"role" : "user", "content" : prompt}
    ]

    stream = anthropic.chat.completions.create(
        model  = "claude-haiku-4-5",
        messgaes = messages,
        stream = True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=display_handle.display_id)
        #yield result


In [56]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):

        """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def slect_relevant_links(url, model):
    response = openai.chat.completions.create(
        model = model,
        messages = [
            {"role" : "system" , "content" : link_system_prompt},
            {"role" : "user" , "content" : get_links_user_prompt(url)}
        ],
        response_format = {"type" : "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    #print(f"Found {len(links['links'])} relevant links")
    return links

def fetch_page_and_relevant_links(url, model):
    contents = fetch_website_contents(url)
    relevant_links = slect_relevant_links(url, model)

    result  = f"## Landing page:\n\n {contents} \n\n ## Relevnt Links: \n"
    for link in relevant_links['links']:
        result += f"\n Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [57]:

def create_brochure_user_prompt(company_name, url, model):
    user_prompt = f"""You are looking at a company called {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt = fetch_page_and_relevant_links(url, model)
    user_prompt = user_prompt[:5000]
    return user_prompt

def stream_brochure(company_name, url, model):
    gpt_model = "gpt-5-mini"
    claude_model = "claude-haiku-4-5"
    yield ""

    if model == "GPT":
        user_prompt = create_brochure_user_prompt(company_name, url, gpt_model)
        result = stream_gpt(user_prompt)
    elif model == "Claude":
        user_prompt = create_brochure_user_prompt(company_name, url, claude_model)
        result = stream_claude(user_prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label="Company name: ")
url_input = gr.Textbox(label = "Landing page url including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response: ");

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT"],
        ["Edward Donner", "https://edwarddonner.com", "Claude"]
    ],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


# Hugging Face: The AI Community Building the Future

---

## Overview

Hugging Face is a leading platform fostering collaboration in the field of artificial intelligence and machine learning. It’s the vibrant community hub where developers, researchers, and enterprises come together to create, share, and improve state-of-the-art machine learning models, datasets, and AI applications. With over 2 million models and half a million datasets hosted, Hugging Face empowers innovation through open collaboration on a truly global scale.

---

## What We Offer

- **Model Hosting & Collaboration:** Easily host, share, and collaborate on unlimited public machine learning models. Access a vast repository with trending models such as MiniMaxAI and Muse-Glimmer-30B.
  
- **Datasets:** A curated collection of over 500,000 datasets including popular ones like SQuAD and ultrachat_200k, continuously updated to support cutting-edge research.

- **Spaces:** Interactive AI apps and demos powered by community contributions enable real-time testing and deployment of models and algorithms, including advanced video and image generation tools.

- **Enterprise Solutions:** Hugging Face PRO offers tailored support, inference endpoints, dedicated infrastructure, and advanced tools to meet business needs.

- **Community & Learning:** Active forums, Discord channels, GitHub repositories, and daily blogs/papers foster knowledge exchange, mentorship, and rapid dissemination of AI advances.

---

## Company Culture

Hugging Face champions openness, collaboration, and innovation. It nurtures a passionate AI community that is inclusive and driven by a shared mission: to advance AI technology ethically and transparently. Contributors from around the world share their work openly, aiding collective progress and empowering developers at every skill level.

The company emphasizes:

- **Community-Driven Growth:** Encouraging open-source contributions and peer collaboration.
- **Transparency & Ethics:** Prioritizing responsible AI practices and open dialogue.
- **Continuous Learning:** Providing rich learning resources and active forums for skill development.
- **Innovation:** Developing cutting-edge tools and adapting to the fast-evolving AI landscape.

---

## Customers & Partners

Hugging Face serves a diverse customer base:

- **Researchers & Academia:** Providing access to models and datasets that accelerate scientific discovery.
- **AI Developers & Startups:** Enabling rapid prototyping and deployment through shared resources.
- **Enterprises:** Offering scalable AI infrastructure and expert support for integration into business workflows.
- **Open Source Communities:** Hosting and promoting collaborative projects with global contributors.

---

## Careers & Opportunities

Hugging Face offers exciting career opportunities for AI researchers, engineers, data scientists, and community managers passionate about shaping the future of machine learning. Employees thrive in a culture that values:

- Creativity and innovation in AI solutions.
- Collaborative teamwork and knowledge sharing.
- Impact-driven projects with a global reach.
- Flexible and inclusive work environments.

Visit Hugging Face’s website to explore current job openings and join a pioneering team committed to building the future of AI.

---

## Join the Future of AI with Hugging Face

Whether you are a developer looking to share or find cutting-edge AI models, a business seeking robust AI solutions, or a researcher eager to collaborate and innovate — Hugging Face is your platform for growth and discovery.

Explore over 2 million models, join an active community, and be part of the AI revolution at [huggingface.co](https://huggingface.co).

---

**Hugging Face**  
*AI community. Open collaboration. Infinite possibilities.*

Traceback (most recent call last):
  File "/Users/swetharaks.jayakumar/petprojects/AI-Engineering-Core-Track/Udemy-EdDonner-MasterAIAndLLMs/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/swetharaks.jayakumar/petprojects/AI-Engineering-Core-Track/Udemy-EdDonner-MasterAIAndLLMs/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/swetharaks.jayakumar/petprojects/AI-Engineering-Core-Track/Udemy-EdDonner-MasterAIAndLLMs/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/swetharaks.jayakumar/petprojects/AI-Engineering-Core-Track/Udemy-EdDonner-MasterAIAndLLMs/.venv/lib/pyth